# FRM · Phase 1 — Teacher Label Generation (GPU, one-time, expensive)

Runs the FROZEN SmolVLM2-2.2B over every gazed WearVQA sample to cache the global tokens `G[64,d]` and the attention-rollout importance labels `imp_answer / imp_question / imp_context`. Resumable — safe to re-run. **Requires a GPU runtime.** Do this before any experiment.

In [ ]:
# 1) mount Google Drive (dataset + outputs live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) clone the repo containing frm/  (EDIT REPO_URL if your repo name differs)
import os
REPO_URL = "https://github.com/shubhamOjha1000/AAAI_2027_code.git"
REPO_DIR = "/content/AAAI_2027_code"
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
# locate frm/ (it sits at the repo root)
FRM = os.path.join(REPO_DIR, "frm")
if not os.path.exists(os.path.join(FRM, 'config.py')):
    import subprocess
    hit = subprocess.check_output(['bash','-lc',
        f"find {REPO_DIR} -name config.py -path '*frm*' | head -1"]).decode().strip()
    FRM = os.path.dirname(hit)
assert os.path.exists(os.path.join(FRM, "config.py")), "frm/ not found — check REPO_URL"
%cd $FRM

In [ ]:
# 3) install deps
!pip -q install -r requirements.txt

In [ ]:
# 4) point at the dataset on Drive + output dir; put frm/ on the path
import os, sys
os.environ["DATA_DIR"] = "/content/drive/MyDrive/wearvqa_gaze_only"
os.environ["FRM_OUT_DIR"] = "/content/drive/MyDrive/frm_out"
sys.path.insert(0, os.getcwd())
import config as C; print('DATA_DIR =', C.DATA_DIR); print('OUT_DIR  =', C.OUT_DIR)

### GPU check

In [ ]:
!nvidia-smi -L

### Smoke test (5 samples) — verify the VLM + rollout path before the full run

In [ ]:
import labels
labels.generate(limit=5)

### Full run (all ~1000 samples). Resumes from where it stopped.

In [ ]:
import importlib, labels; importlib.reload(labels)
labels.generate()

### Peek at cached labels

In [ ]:
from labels import load_labels
metas, G, imp = load_labels()
print('samples:', len(metas), '| G shape:', G.shape)
print('imp_answer[0][:8]:', imp['imp_answer'][0][:8])